# Clock - run everything (NHATS, Bone et al. 2025)
**Spec:** PLAN.md H1-H7, B3. **Runs on:** Colab **GPU** (Runtime > Change runtime type > T4), **no Drive**. NHATS must not go to Kaggle or any AI assistant: **do NOT use Colab's Gemini here**, do not paste or screenshot images.

Upload to the Files panel (`/content`) and check the sizes with `!ls -l /content/*.tar` before running:

| File (from `D:\Alzheimers\colab_upload\`) | Exact size |
|---|---|
| `clock_code.tar` | rebuilt after this notebook was written |
| `clock_data_lean.tar` | 1,044,080,640 bytes |

Then **Runtime > Run all** and keep the tab open. Order: labels + split (about 5 min) > **smoke test** on a few dozen images (about 10 min; stops here if anything is broken) > full run: preprocess (images area-averaged 4x) > Crop-Net on synthetic pages > 3-step crop > ViT head phase > ViT full phase > test metrics > ONNX. Expect roughly 3-5 hours. Every trainer saves checkpoints in `/content/out`; if Colab disconnects, upload the two `.pt` checkpoints back to `/content/out/` and Run all again (data prep is redone, training resumes).

When it ends, the last two cells download `clock_results.zip` (logs, aggregate numbers) and `clock_models.zip` (ONNX + checkpoints). Paste Claude the printed **metrics lines** and the **crop QC line**; nothing else needs to be pasted. Deviations from the paper: FINDINGS 3e.

In [ ]:
# Run in EVERY new session: unpack what you uploaded into /content
import os, sys, subprocess
os.makedirs('/content/Alzheimers', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
subprocess.run(['tar', '-xf', '/content/clock_code.tar', '-C', '/content/Alzheimers'], check=True)
subprocess.run(['tar', '-xf', '/content/clock_data_lean.tar', '-C', '/content/data'], check=True)   # creates /content/data/NHATS
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'], check=True)
ROOT = '/content/Alzheimers'; NHATS = '/content/data/NHATS'; OUT = '/content/out'
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)
import numpy as np, pandas as pd, torch
from clock import labels, split, pipeline
from common.runlog import RunLog
log = RunLog(OUT + '/clock.log')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime > Change runtime type > GPU')

## 1. Labels and split (H1-H2)

In [ ]:
with log.tee('H1 label table'):
    tab = labels.build_label_table(NHATS, NHATS + '/survey')
    labels.summarize(tab)
with log.tee('H2 split'):
    tab = split.split_by_spid(tab, val_images=1000, test_frac=0.10, seed=0)
    split.report(tab)

## 2. Smoke test (a few dozen images, tiny budgets). If this fails, stop and send Claude the last 15 lines

In [ ]:
with log.tee('SMOKE TEST'):
    pipeline.run(pipeline.SMOKE, tab, NHATS)
    print('SMOKE OK')
import gc, shutil; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree('/content/work_smoke', ignore_errors=True)
shutil.rmtree('/content/out_smoke', ignore_errors=True)


## 3a. Full run, part 1: preprocess, Crop-Net, 3-step crop


In [ ]:
with log.tee('FULL RUN part 1'):
    pipeline.run(pipeline.FULL, tab, NHATS, stop_after='crop')


Download the Crop-Net checkpoint now (about 80 MB) so a late failure does not lose it. Click Allow if the browser asks; if it is blocked the run still continues.


In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('/content/cropnet_checkpoint.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/content/out/cropnet_best.pt', 'cropnet_best.pt')
try:
    files.download('/content/cropnet_checkpoint.zip')
except Exception as e:
    print('download skipped:', e)


## 3b. Full run, part 2: ViT head phase, ViT full phase, test metrics, ONNX (earlier stages are skipped)


In [ ]:
with log.tee('FULL RUN part 2'):
    pipeline.run(pipeline.FULL, tab, NHATS)


> **SEND BACK TO CLAUDE:** the lines starting `crop QC`, `threshold`, `balanced accuracy`, `F1`, `AUC`, `clock-score MSE`, `test dementia prevalence`, `ViT ONNX`, `Crop-Net ONNX` and the final `finished in` line. Do not paste images.

## 4. Downloads (two zips). Click Allow if the browser asks. Save to `D:\Alzheimers\output\`

In [ ]:
import shutil, zipfile
from google.colab import files
with zipfile.ZipFile('/content/clock_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('clock.log', 'crop_qc.csv', 'clock_test_preds.csv', 'clock_features.json', 'split_note.txt'):
        if os.path.exists(f'{OUT}/{f}'): z.write(f'{OUT}/{f}', f)
tab.to_csv(f'{OUT}/split.csv', index=False)
with zipfile.ZipFile('/content/clock_results.zip', 'a', zipfile.ZIP_DEFLATED) as z:
    z.write(f'{OUT}/split.csv', 'split.csv')
files.download('/content/clock_results.zip')

In [ ]:
with zipfile.ZipFile('/content/clock_models.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('clock_vit.onnx', 'clock_cropnet.onnx', 'clock_features.json', 'cropnet_best.pt', 'vit_full_best.pt'):
        if os.path.exists(f'{OUT}/{f}'): z.write(f'{OUT}/{f}', f)
print(os.path.getsize('/content/clock_models.zip') / 1e6, 'MB. If the download does not start: Files panel > refresh > right-click clock_models.zip > Download.')
files.download('/content/clock_models.zip')